# 🎤 YuE sur Colab — MusicGen Studio (test)

1. Menu **Exécution → Modifier le type d'exécution → T4 GPU** → **Enregistrer**.
2. Clique sur **▶** de la cellule ci-dessous. Une seule cellule, rien d'autre à faire.
3. Attends le **cadre vert** (15-25 min la 1re fois : compilation + ~13 Go de modèles) et copie le lien `https://….gradio.live`.
4. Colle-le dans **MusicGen Studio → ⚙ Réglages → Lien YuE**, puis choisis le moteur **🎤 YuE (Colab)**.

YuE (1re génération, licence Apache 2.0) chante surtout en **anglais** ; ~30 s de chant = plusieurs minutes sur T4.
Laisse l'onglet ouvert pendant que tu utilises le studio.


In [ ]:
#@title ▶ Lancer YuE (clique sur le bouton ▶ à gauche)
#@markdown Laisse les options par défaut. Tout est automatique : installation, modèles, puis lien vert.
import os, re, subprocess, sys, time, collections, shutil
from IPython.display import HTML, display

COMPAT = '# Correctif YuE : le torchaudio recent confie la lecture/ecriture audio a torchcodec ; repli sur soundfile\ntry:\n    import torch, torchaudio, soundfile as sf\n    try:\n        import torchcodec  # noqa: F401\n    except Exception:\n        def _load(path, *a, **k):\n            data, sr = sf.read(str(path), dtype="float32", always_2d=True)\n            return torch.from_numpy(data.T.copy()), sr\n        def _save(path, wav, sample_rate, *a, **k):\n            w = wav.detach().cpu().float()\n            if w.dim() == 1:\n                w = w.unsqueeze(0)\n            sf.write(str(path), w.numpy().T, int(sample_rate))\n        torchaudio.load, torchaudio.save = _load, _save\nexcept Exception:\n    pass\n'
COMMIT = "3e63b69be31cfc77bf21c7f0793a29cd7617f5e8"   # version de YuE-extend (Mozer) figée : ne pas changer
REPO = "/content/YuE-extend"

def box(msg, ok=True):
    bg, bd = ("#0f3d2a", "#2ecc71") if ok else ("#4a1414", "#e74c3c")
    display(HTML(f'<div style="margin:10px 0;padding:14px 18px;border-radius:10px;background:{bg};'
                 f'border:2px solid {bd};color:#fff;font-family:sans-serif;font-size:15px">{msg}</div>'))

def stop(msg):
    box("❌ " + msg, ok=False)
    raise SystemExit(msg)

def sh(cmd, cwd=None, tries=1):
    for i in range(tries):
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
        if r.returncode == 0:
            return r.stdout
        if i < tries - 1:
            print(f"   nouvel essai ({i + 2}/{tries})…"); time.sleep(10)
    print(r.stdout[-1500:]); print(r.stderr[-3000:])
    stop("Étape échouée : " + cmd.split()[0] + ". Envoie une capture de ce message.")

# 1) GPU
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip()
if not gpu:
    stop("Pas de GPU. Menu <b>Exécution → Modifier le type d'exécution → T4 GPU</b> → Enregistrer, puis relance ▶.")
print("✅ 1/5 GPU :", gpu)

# 2) Code (version figée)
if not os.path.isdir(REPO + "/.git"):
    shutil.rmtree(REPO, ignore_errors=True)
    sh(f"git init -q {REPO} && cd {REPO} && git remote add origin https://github.com/Mozer/YuE-extend.git")
    sh(f"git fetch -q --depth 1 origin {COMMIT} && git checkout -q FETCH_HEAD", cwd=REPO, tries=3)
print("✅ 2/5 Code YuE prêt")

# 3) Dépendances (torch et gradio de Colab conservés ; flash-attn inutile sur T4)
if not os.path.exists("/content/.yue_deps_v2"):
    print("⏳ 3/5 Installation des dépendances (3 à 5 min)…")
    req = [l.strip() for l in open(REPO + "/requirements.txt") if l.strip()]
    req = [l for l in req if not re.match(r"(torch|torchaudio|gradio|flash-attn|sageattention)\b", l)]
    sh("pip install -q exllamav2 ninja " + " ".join(f"'{l}'" for l in req), tries=3)
    sh("pip install -q --upgrade protobuf && pip uninstall -q -y flash-attn", tries=2)
    # Python 3.13 de Colab : transformers trop récent pour YuE -> version 2025 dans un dossier réservé à YuE
    sh("pip install -q --no-deps --target /content/yue_libs transformers==4.51.3 tokenizers==0.21.4 "
       "huggingface_hub==0.34.4", tries=3)
    os.makedirs("/content/yue_compat", exist_ok=True)
    open("/content/yue_compat/sitecustomize.py", "w").write(COMPAT)
    open("/content/.yue_deps_v2", "w").write("ok")
print("✅ 3/5 Dépendances installées")

# 4) Modèles (versions compressées EXL2 8 bits : tiennent dans les 15 Go du T4)
print("⏳ 4/5 Téléchargement des modèles (~13 Go, 5 à 10 min la 1re fois)…")
from huggingface_hub import snapshot_download
for rid, dst in (("m-a-p/xcodec_mini_infer", REPO + "/xcodec_mini_infer"),
                 ("Alissonerdx/YuE-s1-7B-anneal-en-cot-exl2-8.0bpw", REPO + "/workspace/models/YuE-s1-7B-anneal-en-cot-exl2-8.0bpw"),
                 ("Alissonerdx/YuE-s2-1B-general-exl2-8.0bpw", REPO + "/workspace/models/YuE-s2-1B-general-exl2-8.0bpw")):
    for k in range(3):
        try:
            snapshot_download(rid, local_dir=dst); break
        except Exception as e:
            if k == 2: stop(f"Téléchargement de {rid} impossible : {e}")
            time.sleep(10)
# exllamav2 récent ne lit plus que tokenizer.json : conversion de tokenizer.model (identifiants identiques)
open("/content/yue_tok.py", "w").write(chr(10).join([
    "import glob, os",
    "from transformers import LlamaTokenizerFast",
    "for d in [os.path.dirname(f) for f in glob.glob('/content/YuE-extend/workspace/models/*/tokenizer.model')]:",
    "    if not os.path.exists(d + '/tokenizer.json'):",
    "        LlamaTokenizerFast(vocab_file=d + '/tokenizer.model', from_slow=True, legacy=True).save_pretrained(d)", ""]))
sh("PYTHONPATH=/content/yue_libs python /content/yue_tok.py", tries=2)
# Compile l'accélérateur exllamav2 maintenant (sinon la 1re génération attend 5-10 min de plus)
print("⏳    Compilation d'exllamav2 (5 à 10 min la 1re fois)…")
sh('python -c "import exllamav2, exllamav2.ext"', tries=1)
print("✅ 4/5 Modèles prêts")

# 5) Serveur + lien
SERVER = '\nimport os, re, glob, shutil, subprocess, threading, time, random\nimport gradio as gr\n\nREPO = "/content/YuE-extend"\nM = REPO + "/workspace/models"\nX = REPO + "/xcodec_mini_infer"\nLOCK = threading.Lock()\n# transformers 4.51 (YuE date de 2025) + correctif audio, reserves aux calculs YuE (Gradio garde les versions de Colab)\nYENV = dict(os.environ, PYTHONPATH="/content/yue_libs:/content/yue_compat:" + os.environ.get("PYTHONPATH", ""))\n\ndef norm_lyrics(text):\n    secs, cur = [], None\n    for line in (text or "").splitlines():\n        m = re.fullmatch(r"\\s*\\[([^\\]]+)\\]\\s*", line)\n        if m:\n            cur = [f"[{re.sub(r\'[^a-z]\', \'\', m.group(1).lower()) or \'verse\'}]"]\n            secs.append(cur)\n        elif line.strip():\n            if cur is None:\n                cur = ["[verse]"]; secs.append(cur)\n            cur.append(line.strip())\n    return [s for s in secs if len(s) > 1]\n\ndef yue_generate(genre, lyrics, duration, seed):\n    secs = norm_lyrics(lyrics)\n    if not secs:\n        raise gr.Error("YuE a besoin de paroles (sections [verse], [chorus]…).")\n    n = max(1, min(len(secs), round(float(duration) / 30), 4))\n    seed = int(seed) if int(seed) >= 0 else random.randint(0, 2**31 - 1)\n    with LOCK:\n        job = f"/content/yue_jobs/{int(time.time())}"\n        os.makedirs(job, exist_ok=True)\n        open(job + "/genre.txt", "w", encoding="utf-8").write(" ".join((genre or "pop").replace(",", " ").split()))\n        open(job + "/lyrics.txt", "w", encoding="utf-8").write("\\n\\n".join("\\n".join(s) for s in secs))\n        cmd = ["python", "-u", "src/yue/infer.py",\n               "--stage1_use_exl2", "--stage1_model", M + "/YuE-s1-7B-anneal-en-cot-exl2-8.0bpw",\n               "--stage1_cache_size", "30000", "--stage1_cache_mode", "FP16",\n               "--stage2_use_exl2", "--stage2_model", M + "/YuE-s2-1B-general-exl2-8.0bpw",\n               "--stage2_cache_size", "50000", "--stage2_cache_mode", "FP16",\n               "--genre_txt", job + "/genre.txt", "--lyrics_txt", job + "/lyrics.txt",\n               "--run_n_segments", str(n), "--max_new_tokens", "3000", "--output_dir", job + "/out",\n               "--cuda_idx", "0", "--seed", str(seed), "--resume_after_n", "-1",\n               "--basic_model_config", X + "/final_ckpt/config.yaml", "--resume_path", X + "/final_ckpt/ckpt_00360000.pth",\n               "--config_path", X + "/decoders/config.yaml", "--vocal_decoder_path", X + "/decoders/decoder_131000.pth",\n               "--inst_decoder_path", X + "/decoders/decoder_151000.pth", "--no_flash_attn"]\n        r = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True, env=YENV)\n        open(job + "/log.txt", "w").write(r.stdout + "\\n" + r.stderr)\n        mp3 = sorted(glob.glob(job + "/out/*.mp3"), key=os.path.getmtime)\n        if r.returncode != 0 or not mp3:\n            err = [l for l in (r.stdout + r.stderr).splitlines() if l.strip() and "absl::" not in l and "cudart_stub" not in l][-6:]\n            raise gr.Error("YuE a échoué : " + " | ".join(err)[-600:])\n        return mp3[-1]\n\nwith gr.Blocks(title="YuE pour MusicGen Studio") as demo:\n    gr.Markdown("### YuE — serveur pour MusicGen Studio")\n    g = gr.Textbox(label="Genre (tags)", value="pop upbeat female bright vocal")\n    l = gr.Textbox(label="Paroles", lines=10, value="[verse]\\nHello world\\n\\n[chorus]\\nSing along")\n    d = gr.Number(label="Durée (s)", value=30)\n    s = gr.Number(label="Graine (-1 = hasard)", value=-1)\n    a = gr.Audio(label="Résultat", type="filepath")\n    gr.Button("Générer").click(yue_generate, [g, l, d, s], a, api_name="yue_generate")\n\ndemo.queue(default_concurrency_limit=1).launch(share=True, server_name="127.0.0.1", server_port=7861,\n                                                allowed_paths=["/content/yue_jobs"])\n'
open("/content/yue_server.py", "w", encoding="utf-8").write(SERVER)
subprocess.run("pkill -f yue_server.py", shell=True)
env = {k: v for k, v in os.environ.items() if k not in ("PYTHONHOME", "VIRTUAL_ENV", "MPLBACKEND")}
env.update(PYTHONUNBUFFERED="1", GRADIO_ANALYTICS_ENABLED="False", MPLBACKEND="Agg")
print("⏳ 5/5 Démarrage du serveur…")
proc = subprocess.Popen([sys.executable, "/content/yue_server.py"], cwd=REPO, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)

tail = collections.deque(maxlen=60)
shown = False
KEEP = ("Error", "ERROR", "Traceback", "out of memory")
try:
    for line in proc.stdout:
        tail.append(line.rstrip())
        m = re.search(r"https://[a-z0-9-]+\.gradio\.live", line)
        if m and not shown:
            shown = True
            box(f'✅ <b>YuE est prêt.</b> Copie ce lien dans <b>MusicGen Studio → ⚙ Réglages → Lien YuE</b> :'
                f'<div style="font-size:22px;font-weight:700;margin-top:8px;user-select:all;word-break:break-all">{m.group(0)}</div>'
                f'<div style="font-size:12px;margin-top:8px;opacity:.8">Le lien marche tant que cette cellule tourne '
                f'(icône qui tourne). Ne ferme pas cet onglet.</div>')
        elif any(k in line for k in KEEP):
            print(line, end="")
except KeyboardInterrupt:
    proc.terminate()
    print("⏹ Serveur arrêté.")

code = proc.wait()
if code not in (0, None, -15):
    print("\n".join(list(tail)[-30:]))
    box(f"⚠️ Le serveur s'est arrêté (code {code}). Envoie une capture de ce message, "
        "ou relance ▶ pour obtenir un nouveau lien.", ok=False)
